# 15SW Safety — Watcher: weather at the time

Links **every mishap to the weather at its time and place**, then checks whether bad
weather shows up on our mishap days **more often than on ordinary days**. The result
appears in the **Predictive Safety Forecast** panel on the app's **Forecast** page.

### How to run
**Runtime → Run all.** The first run downloads about 10 years of airfield reports
(3–6 minutes). It prints a summary and saves it to the app.

### First time only (Google Colab)
Use the same two secrets as the forecast notebook (key icon in the left bar, *Notebook access* on):
- `APP_URL` — the live site, e.g. `https://safety.15thstrikewing.mil.ph`
- `MODEL_API_TOKEN` — the same value as `MODEL_API_TOKEN` in the app's `.env`

The app sends only what this needs: mishap id, date, time, location, flight/ground,
type, category and aircraft. It never sends descriptions or names. On a laptop without
the secrets, it reads and writes the local database instead.

### Where the weather comes from
| | Source | Used for |
|---|---|---|
| **Observed** | Airfield METAR reports (Iowa State University archive) | Places with an airfield within the distance you set |
| **Estimate** | Weather model (ERA5 reanalysis via Open-Meteo) | Places with no airfield nearby, e.g. Lumbia / CDO, Cotabato |

Only **observed** weather goes into the mishap-day vs ordinary-day comparison.

> **Weather present ≠ weather caused it.** This shows what the weather *was*. The
> safety board decides the cause.


In [1]:
# @title 📍 Places → map position (edit if a place is missing or wrong)
# Each mishap "Location" is matched to a place here, then to the nearest airfield report.
# Add new places as "NAME": (latitude, longitude). Names match whole words, ignoring case.
PLACES = {
    # Luzon — Cavite / Metro Manila
    "MDAAB": (14.4950, 120.9060),                  # Danilo Atienza Air Base, Sangley Point
    "SANGLEY": (14.4950, 120.9060),
    "CAVITE CITY": (14.4791, 120.8970),
    "KAWIT": (14.4447, 120.9039),
    "TANZA": (14.3944, 120.8531),
    "GEN TRIAS": (14.3869, 120.8817),
    "CENTENNIAL ROAD": (14.4333, 120.8870),
    "PASAY CITY": (14.5378, 121.0014),
    "MACAPAGAL BLVD": (14.5350, 120.9880),
    "MARILAO": (14.7578, 120.9483),
    "TAGATAY-CALAMBA ROAD": (14.1500, 121.0500),
    "TAGAYTAY-CALAMBA ROAD": (14.1500, 121.0500),
    "TANAUAN": (14.0860, 121.1000),
    "TANUAN": (14.0860, 121.1000),
    "SAN JUAN, BATANGAS": (13.8260, 121.3960),
    "FAB": (13.9550, 121.1250),                    # Fernando Air Base, Lipa
    # Luzon — Central / North
    "CAB": (15.1860, 120.5600),                    # Clark Air Base
    "CVGR": (15.3600, 120.4300),                   # Crow Valley Gunnery Range, Tarlac
    "PILAR, BATAAN": (14.6600, 120.5650),
    "SAN FERNANDO AIRPORT": (16.5956, 120.3031),   # La Union
    "TOG 5": (13.1575, 123.7350),                  # Legazpi
    # Visayas
    "TOG 8": (11.2276, 125.0277),                  # Tacloban
    "GETAFE": (10.1500, 124.1500),                 # Bohol
    # Mindanao
    "EAAB": (6.9224, 122.0596),                    # Edwin Andrews Air Base, Zamboanga
    "TOG 9": (6.9224, 122.0596),                   # ASSUMED at Zamboanga (EAAB) — change if not
    "ZAMBOANGA DEL NORTE": (8.1000, 122.9000),
    "KHTB": (6.0520, 121.0020),                    # Kuta Heneral Teodulfo Bautista, Jolo
    "JOLO": (6.0520, 121.0020),
    "TOG 10": (8.4150, 124.6110),                  # ASSUMED at Lumbia, Cagayan de Oro — change if not
    "LAB": (8.4150, 124.6110),                     # Lumbia Air Base
    "LUMBIA": (8.4150, 124.6110),
    "CAGAYAN DE ORO": (8.4542, 124.6319),
    "MALAYBALAY": (8.1575, 125.1278),
    "MALABAYBAY": (8.1575, 125.1278),
    "BUKIDNON": (8.0500, 125.0000),
    "BUTUAN": (8.9475, 125.5406),
    "DAVAO DE ORO": (7.5500, 126.0000),
    "TOG 12": (7.1652, 124.2096),                  # Cotabato (Awang airport)
}

# Airfields with a long METAR archive (Iowa State University, network PH__ASOS).
AIRFIELDS = {
    "RPLL": ("Manila", 14.5069, 121.0042),
    "RPLC": ("Clark", 15.1667, 120.5667),
    "RPLB": ("Subic Bay", 14.7944, 120.2714),
    "RPLI": ("Laoag", 18.1781, 120.5315),
    "RPVM": ("Mactan-Cebu", 10.3083, 123.9784),
    "RPVD": ("Dumaguete", 9.3335, 123.2973),
    "RPVK": ("Kalibo", 11.6794, 122.3763),
    "RPVP": ("Puerto Princesa", 9.7420, 118.7590),
    "RPMZ": ("Zamboanga", 6.9205, 122.0631),
    "RPMD": ("Davao", 7.1255, 125.6458),
    "RPMR": ("General Santos", 6.0580, 125.0960),
}
print(f"{len(PLACES)} places, {len(AIRFIELDS)} airfields ready.")


38 places, 11 airfields ready.


In [2]:
# @title ▶ Run the weather watcher
# @markdown Set the options, then press ▶. The first run takes 3–6 minutes; a summary prints at the end.

# @markdown **Furthest airfield report to trust (km).** Thunderstorms are local, so keep this small:
MAX_DISTANCE_KM = 50  # @param {type:"slider", min:10, max:150, step:5}
# @markdown **No airfield that close?** Use a weather-model estimate (labelled "estimate", never counted in the comparison):
USE_MODEL_ESTIMATES = True  # @param {type:"boolean"}
# @markdown **Hours either side of the mishap time**, when a time is recorded:
TIME_WINDOW_HOURS = 2  # @param {type:"slider", min:1, max:4, step:1}

import os, re, json, math, sqlite3
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy.stats import binomtest

# ── Where the data lives (same link as the forecast notebook) ──────────────
def _setting(name):
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata   # only exists inside Colab
            value = userdata.get(name) or ""
        except Exception:
            pass
    return value.strip()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

APP_URL = _setting("APP_URL").rstrip("/")
MODEL_API_TOKEN = _setting("MODEL_API_TOKEN")
ONLINE = bool(APP_URL and MODEL_API_TOKEN)
if IN_COLAB and not ONLINE:
    raise RuntimeError("Add APP_URL and MODEL_API_TOKEN under Secrets (key icon, left bar), "
                       "turn on Notebook access for both, then press ▶ again.")

def api(method, path, **kwargs):
    r = requests.request(
        method, f"{APP_URL}/api/model/{path}", timeout=180,
        headers={"Authorization": f"Bearer {MODEL_API_TOKEN}", "X-Model-Token": MODEL_API_TOKEN,
                 "Accept": "application/json"},
        **kwargs)
    if not r.ok:
        raise RuntimeError(f"App link {method} {path} failed ({r.status_code}): {r.text[:300]}")
    return r.json()

def find_db():
    for p in [Path("../database/database.sqlite"), Path("database/database.sqlite")]:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError("No APP_URL / MODEL_API_TOKEN set and no local database/database.sqlite found.")

if "PLACES" not in globals():
    raise RuntimeError("Run the 📍 Places cell first (or Runtime → Run all).")

TZ = "Asia/Manila"
TODAY = pd.Timestamp.now(tz=TZ).tz_localize(None).normalize()
CACHE = Path("data_cache")
MIN_REPORTS = 4          # a day needs at least this many airfield reports to count
UA = {"User-Agent": "15SW-Safety-watcher/1.0 (Wing Safety Office)"}

# The same hazard rules as the live airfield weather on the Forecast page.
BRIEF_KEYS = ["ts", "lowvis", "gusty"]
COMPARE = [("brief", "Any brief-level weather"), ("ts", "Thunderstorms"),
           ("lowvis", "Low visibility (under 5 km)")]   # gusts still count inside "Any brief-level weather"

# ── Inputs ─────────────────────────────────────────────────────────────────
def load_mishaps():
    if ONLINE:
        rec = pd.DataFrame(api("GET", "data")["mishaps"])
    else:
        con = sqlite3.connect(find_db())
        try:
            rec = pd.read_sql_query("SELECT id, mishap_date, mishap_time, location, environment, "
                                    "mishap_type, category, aircraft FROM mishaps", con)
        finally:
            con.close()
    rec["mishap_date"] = pd.to_datetime(rec["mishap_date"]).dt.normalize()
    rec["mishap_time"] = rec["mishap_time"].where(rec["mishap_time"].notna(), None)
    return rec.sort_values(["mishap_date", "id"]).reset_index(drop=True)

# ── Places and airfields ───────────────────────────────────────────────────
def norm(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().upper()

PLACE_KEYS = sorted(((norm(k), v) for k, v in PLACES.items()), key=lambda kv: -len(kv[0]))

def place_of(location):
    loc = norm(location)
    if not loc:
        return None
    for key, pos in PLACE_KEYS:
        if loc == key or re.search(r"(?<![A-Z0-9])" + re.escape(key) + r"(?![A-Z0-9])", loc):
            return key, pos
    return None

def km(a, b):
    (la1, lo1), (la2, lo2) = a, b
    p1, p2 = math.radians(la1), math.radians(la2)
    h = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lo2 - lo1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))

def nearest_airfield(pos):
    sid = min(AIRFIELDS, key=lambda s: km(pos, AIRFIELDS[s][1:]))
    return sid, round(km(pos, AIRFIELDS[sid][1:]))

# ── Observed weather: airfield METAR archive ───────────────────────────────
IEM = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

def metar_year(station, year):
    """One station-year of METARs, cached on disk (past years never change)."""
    CACHE.mkdir(exist_ok=True)
    stamp = f"_{TODAY:%Y%m%d}" if year == TODAY.year else ""
    f = CACHE / f"metar_{station}_{year}{stamp}.csv"
    if not f.exists():
        end = min(pd.Timestamp(year + 1, 1, 1), TODAY + pd.Timedelta(days=1))
        r = requests.get(IEM, headers=UA, timeout=300, params={
            "station": station, "data": ["vsby", "sknt", "gust", "wxcodes", "metar"],
            "year1": year, "month1": 1, "day1": 1,
            "year2": end.year, "month2": end.month, "day2": end.day,
            "tz": "Etc/UTC", "format": "onlycomma", "missing": "M", "trace": "T", "latlon": "no"})
        r.raise_for_status()
        f.write_text(r.text, encoding="utf-8")
    return pd.read_csv(f, na_values=["M", "T"], dtype={"wxcodes": str, "metar": str})

def metar_archive(station, first_year):
    frames = []
    for year in range(first_year, TODAY.year + 1):
        try:
            frames.append(metar_year(station, year))
        except Exception as e:
            print(f"     {station} {year}: skipped ({e.__class__.__name__})")
    if not frames:
        return pd.DataFrame()
    m = pd.concat(frames, ignore_index=True)
    for c in ("vsby", "sknt", "gust"):
        m[c] = pd.to_numeric(m[c], errors="coerce")
    m["local"] = pd.to_datetime(m["valid"]) + pd.Timedelta(hours=8)     # UTC → Philippine time
    m["day"] = m["local"].dt.normalize()
    wx = m["wxcodes"].fillna("").str.upper()
    raw = " " + m["metar"].fillna("").str.upper() + " "
    m["ts"] = wx.str.contains("TS") | raw.str.contains(r"\bLTG", regex=True)
    m["cb"] = raw.str.contains(r"\dCB\b|\sCB\s", regex=True) & ~m["ts"]
    m["lowvis"] = m["vsby"] < 3                       # statute miles (≈ 4.8 km)
    m["gusty"] = m["gust"] >= 25
    m["windy"] = (m["sknt"] >= 20) & ~m["gusty"]
    m["haze"] = wx.str.contains("HZ|FU|BR|FG")
    m["rain"] = wx.str.contains("RA|SH") & ~wx.str.contains("TS")
    return m

def span(times):
    a, b = times.min(), times.max()
    return f"{a:%H:%M}" if a == b else f"{a:%H:%M}–{b:%H:%M}"

def observed_hazards(rows):
    out = []
    if rows["ts"].any():
        out.append({"text": f"Thunderstorms ({span(rows.loc[rows['ts'], 'local'])})", "level": "brief"})
    elif rows["cb"].any():
        out.append({"text": "CB clouds nearby", "level": "aware"})
    if rows["lowvis"].any():
        out.append({"text": f"Low visibility ({rows['vsby'].min() * 1.609:.1f} km)", "level": "brief"})
    if rows["gusty"].any():
        out.append({"text": f"Gusts {int(rows['gust'].max())} kt", "level": "brief"})
    elif rows["windy"].any():
        out.append({"text": f"Strong wind {int(rows['sknt'].max())} kt", "level": "aware"})
    if rows["haze"].any():
        out.append({"text": "Haze / mist / smoke", "level": "aware"})
    if rows["rain"].any():
        out.append({"text": "Rain / showers", "level": "aware"})
    return out

# ── Estimated weather: ERA5 reanalysis via Open-Meteo ──────────────────────
OPEN_METEO = "https://archive-api.open-meteo.com/v1/archive"

def model_hours(pos, day):
    CACHE.mkdir(exist_ok=True)
    f = CACHE / f"model_{pos[0]:.3f}_{pos[1]:.3f}_{day:%Y%m%d}.json"
    if f.exists():
        data = json.loads(f.read_text())
    else:
        r = requests.get(OPEN_METEO, headers=UA, timeout=60, params={
            "latitude": pos[0], "longitude": pos[1], "start_date": f"{day:%Y-%m-%d}", "end_date": f"{day:%Y-%m-%d}",
            "hourly": "weather_code,precipitation,wind_speed_10m,wind_gusts_10m",
            "timezone": TZ, "wind_speed_unit": "kn"})
        r.raise_for_status()
        data = r.json()
        if all(v is not None for v in data.get("hourly", {}).get("weather_code", [None])):
            f.write_text(json.dumps(data))          # only cache complete days
    h = pd.DataFrame(data.get("hourly", {}))
    if h.empty:
        return h
    h["local"] = pd.to_datetime(h["time"])
    return h.dropna(subset=["weather_code"])

def model_hazards(h):
    out = []
    code = h["weather_code"]
    if (code >= 95).any():
        out.append({"text": f"Thunderstorms ({span(h.loc[code >= 95, 'local'])}, model)", "level": "brief"})
    if (h["wind_gusts_10m"] >= 25).any():
        out.append({"text": f"Gusts {int(h['wind_gusts_10m'].max())} kt (model)", "level": "brief"})
    elif (h["wind_speed_10m"] >= 20).any():
        out.append({"text": f"Strong wind {int(h['wind_speed_10m'].max())} kt (model)", "level": "aware"})
    heavy = (h["precipitation"] >= 7.6) | code.isin([65, 67, 82])
    if heavy.any():
        out.append({"text": "Heavy rain (model)", "level": "aware"})
    elif code.between(51, 67).any() or code.between(80, 82).any():
        out.append({"text": "Rain / showers (model)", "level": "aware"})
    return out

# ── Each mishap ────────────────────────────────────────────────────────────
def level_of(hazards, has_data):
    levels = {h["level"] for h in hazards}
    return "brief" if "brief" in levels else "aware" if "aware" in levels else "clear" if has_data else "no_data"

def window_of(row):
    """(rows mask function, 'day'|'time', label) for this mishap."""
    t = row["mishap_time"]
    if t and re.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?", str(t)):
        hh, mm = map(int, str(t).split(":")[:2])
        at = row["mishap_date"] + pd.Timedelta(hours=hh, minutes=mm)
        w = pd.Timedelta(hours=TIME_WINDOW_HOURS)
        return (lambda df: (df["local"] >= at - w) & (df["local"] <= at + w)), "time", f"±{TIME_WINDOW_HOURS} h of {hh:02d}:{mm:02d}"
    return (lambda df: df["local"].dt.normalize() == row["mishap_date"]), "day", "that day"

def link(row, metars):
    base = {"mishap_id": int(row["id"]), "station": None, "station_name": None, "distance_km": None,
            "source": "none", "window": "day", "level": "no_data", "hazards": [], "note": None, "reports": 0}
    found = place_of(row["location"])
    if not found:
        return base | {"note": "Location not on the places list — add it in the Places cell."}
    key, pos = found
    sid, dist = nearest_airfield(pos)
    pick, window, label = window_of(row)

    if dist <= MAX_DISTANCE_KM:
        m = metars.get(sid)
        rows = m[pick(m)] if m is not None and not m.empty else pd.DataFrame()
        hz = observed_hazards(rows) if len(rows) else []
        note = (f"{len(rows)} reports {label}"
                + (f" · lowest visibility {rows['vsby'].min() * 1.609:.0f} km" if len(rows) and rows["vsby"].notna().any() else "")
                + (f" · strongest wind {int(rows['sknt'].max())} kt" if len(rows) and rows["sknt"].notna().any() else "")
                ) if len(rows) else f"No {sid} reports {label}."
        return base | {"station": sid, "station_name": AIRFIELDS[sid][0], "distance_km": dist, "source": "observed",
                       "window": window, "level": level_of(hz, len(rows) > 0), "hazards": hz,
                       "note": note, "reports": int(len(rows))}

    far = f"Nearest airfield report is {AIRFIELDS[sid][0]} ({dist} km)"
    if not USE_MODEL_ESTIMATES:
        return base | {"note": f"{far} — too far to trust."}
    try:
        h = model_hours(pos, row["mishap_date"])
        h = h[pick(h)] if not h.empty else h
    except Exception as e:
        return base | {"note": f"{far}; weather model unavailable ({e.__class__.__name__})."}
    if h.empty:
        return base | {"note": f"{far}; model data not ready yet (about 5 days' delay)."}
    hz = model_hazards(h)
    return base | {"station_name": "Weather model (ERA5)", "distance_km": 0, "source": "estimate",
                   "window": window, "level": level_of(hz, True), "hazards": hz,
                   "note": f"{far}, so this is a model estimate {label}.", "reports": int(len(h))}

# ── Mishap days vs ordinary days (observed only) ───────────────────────────
def daily_flags(m, start):
    d = m.groupby("day").agg(reports=("valid", "size"), ts=("ts", "any"), lowvis=("lowvis", "any"),
                             gusty=("gusty", "any"))
    d["brief"] = d[BRIEF_KEYS].any(axis=1)
    return d[(d["reports"] >= MIN_REPORTS) & (d.index >= start) & (d.index < TODAY)]

def compare(rec, links, daily):
    obs = rec.merge(links[links["source"] == "observed"][["mishap_id", "station"]],
                    left_on="id", right_on="mishap_id")
    # One row per airfield-day, so three mishaps on the same day count once.
    obs = obs.drop_duplicates(["station", "mishap_date", "environment"])
    rows, detail = [], []
    for group in ("flight", "all"):
        sub = obs if group == "all" else obs[obs["environment"] == "flight"]
        sub = sub.drop_duplicates(["station", "mishap_date"])
        flags = [daily[s].loc[d] if s in daily and d in daily[s].index else None
                 for s, d in zip(sub["station"], sub["mishap_date"])]
        keep = [f is not None for f in flags]
        sub, flags = sub[keep], [f for f in flags if f is not None]
        for key, label in COMPARE:
            n = len(flags)
            k = int(sum(bool(f[key]) for f in flags))
            # "Usual" = the same airfields on all days, weighted by where our mishaps happened.
            p0 = float(np.mean([daily[s][key].mean() for s in sub["station"]])) if n else 0.0
            if n < 10:
                verdict, p = "too few to tell", None
            else:
                q = min(max(p0, 1e-9), 1 - 1e-9)
                hi = binomtest(k, n, q, alternative="greater").pvalue
                lo = binomtest(k, n, q, alternative="less").pvalue
                verdict = "higher than usual" if hi < 0.05 else "lower than usual" if lo < 0.05 else "same as usual"
                p = min(hi, lo)
            rows.append({"group": group, "hazard": label, "mishap_hits": k, "mishap_days": n,
                         "mishap_pct": round(100 * k / n, 1) if n else 0.0, "usual_pct": round(100 * p0, 1),
                         "verdict": verdict})
            detail.append(p)
    return rows, detail

# ── Save ───────────────────────────────────────────────────────────────────
def save(summary, links, source):
    if ONLINE:
        return api("POST", "weather-links", json={"source": source, "summary": summary, "links": links})["saved"]
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    con = sqlite3.connect(find_db())
    try:
        con.execute("DELETE FROM mishap_weather")
        for l in links:
            con.execute("INSERT INTO mishap_weather (mishap_id, station, station_name, distance_km, source, window, "
                        "level, hazards, note, reports, created_at, updated_at) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
                        (l["mishap_id"], l["station"], l["station_name"], l["distance_km"], l["source"], l["window"],
                         l["level"], json.dumps(l["hazards"]), l["note"], l["reports"], now, now))
        con.execute("INSERT INTO watcher_reports (kind, payload, source, generated_at, created_at, updated_at) "
                    "VALUES ('weather_link', ?, ?, ?, ?, ?) ON CONFLICT(kind) DO UPDATE SET payload=excluded.payload, "
                    "source=excluded.source, generated_at=excluded.generated_at, updated_at=excluded.updated_at",
                    (json.dumps(summary), source, now, now, now))
        con.commit()
    finally:
        con.close()
    return len(links)

# ── Run ────────────────────────────────────────────────────────────────────
def run():
    global RECORDS, LINKS, DAILY, SUMMARY
    print("15SW Safety - weather watcher")
    print(f"  Using: {APP_URL if ONLINE else 'the local database (offline)'}")
    rec = load_mishaps()
    print(f"  Records: {len(rec)} mishaps ({int((rec['environment'] == 'flight').sum())} flight), "
          f"{int(rec['mishap_time'].notna().sum())} with a time")

    places = rec["location"].map(place_of)
    unmapped = sorted({str(l).strip() or "no location given" for l, p in zip(rec["location"].fillna(""), places)
                       if not isinstance(p, tuple)})
    near = {nearest_airfield(p[1])[0] for p in places.dropna() if nearest_airfield(p[1])[1] <= MAX_DISTANCE_KM}
    print(f"  Places: {int(places.notna().sum())} matched, {len(unmapped)} not on the list"
          + (f" ({', '.join(unmapped)})" if unmapped else ""))

    start = rec["mishap_date"].min()
    print(f"  Downloading airfield reports since {start.year} (Iowa State METAR archive): {', '.join(sorted(near))}")
    metars, daily = {}, {}
    for sid in sorted(near):
        metars[sid] = metar_archive(sid, start.year)
        if not metars[sid].empty:
            daily[sid] = daily_flags(metars[sid], start)
        print(f"     {sid} {AIRFIELDS[sid][0]}: {len(metars[sid]):,} reports")

    links = [link(r, metars) for _, r in rec.iterrows()]
    L = pd.DataFrame(links)
    est = int((L["source"] == "estimate").sum())
    if est:
        print(f"  Weather-model estimates for {est} mishaps with no airfield within {MAX_DISTANCE_KM} km (Open-Meteo)")

    rows, pvalues = compare(rec, L, daily)
    counts = {"mishaps": len(rec), "observed": int((L["source"] == "observed").sum()), "estimate": est,
              "none": int((L["source"] == "none").sum()), "unmapped": int(places.isna().sum()),
              "with_time": int(rec["mishap_time"].notna().sum())}
    summary = {"period": f"{start:%b %Y} – {TODAY:%b %Y}", "max_distance_km": MAX_DISTANCE_KM,
               "counts": counts, "unmapped": unmapped, "rows": rows}
    saved = save(summary, links, source=f"Watcher notebook · airfields within {MAX_DISTANCE_KM} km")

    print()
    for group, title in (("flight", "FLIGHT MISHAPS"), ("all", "ALL MISHAPS (flight + ground)")):
        print(f"WEATHER ON {title}: mishap days vs ordinary days at the same airfields (observed)")
        for r, p in zip(rows, pvalues):
            if r["group"] == group:
                chance = f"  (p = {p:.2f})" if p is not None else ""
                print(f"  {r['hazard']:<30} {r['mishap_pct']:5.1f}% of {r['mishap_days']} mishap days · "
                      f"usual {r['usual_pct']:5.1f}% · {r['verdict']}{chance}")
        print()
    print("LATEST MISHAPS")
    for r, l in list(zip(rec.itertuples(), links))[-6:][::-1]:
        where = (f"{l['station_name']} {l['station'] or ''} {l['distance_km']} km" if l["source"] == "observed"
                 else l["station_name"] or "no weather")
        hz = ", ".join(h["text"] for h in l["hazards"]) or {"clear": "no hazards", "no_data": "no data"}.get(l["level"], "")
        place = r.location if isinstance(r.location, str) and r.location.strip() else "-"
        print(f"  {r.mishap_date:%d %b %Y}  {place[:18]:<18} {where:<28} {hz}")
    print()
    print(f"Saved weather for {saved} mishaps to {APP_URL or 'the local database'}.")
    if ONLINE:
        print(f"Open the Forecast page: {APP_URL}/forecast")
    RECORDS, LINKS, DAILY, SUMMARY = rec, L, daily, summary

run()


15SW Safety - weather watcher
  Using: the local database (offline)
  Records: 128 mishaps (85 flight), 0 with a time
  Places: 127 matched, 1 not on the list (no location given)


     RPLB Subic Bay: 93,688 reports


     RPLC Clark: 84,487 reports


     RPLL Manila: 94,990 reports


     RPMZ Zamboanga: 67,251 reports


     RPVM Mactan-Cebu: 93,742 reports


  Weather-model estimates for 32 mishaps with no airfield within 50 km (Open-Meteo)

WEATHER ON FLIGHT MISHAPS: mishap days vs ordinary days at the same airfields (observed)
  Any brief-level weather         25.9% of 54 mishap days · usual  30.1% · same as usual  (p = 0.31)
  Thunderstorms                   24.1% of 54 mishap days · usual  25.1% · same as usual  (p = 0.50)
  Low visibility (under 5 km)      3.7% of 54 mishap days · usual   6.8% · same as usual  (p = 0.28)

WEATHER ON ALL MISHAPS (flight + ground): mishap days vs ordinary days at the same airfields (observed)
  Any brief-level weather         24.4% of 90 mishap days · usual  31.0% · same as usual  (p = 0.11)
  Thunderstorms                   21.1% of 90 mishap days · usual  25.5% · same as usual  (p = 0.20)
  Low visibility (under 5 km)      5.6% of 90 mishap days · usual   7.1% · same as usual  (p = 0.37)

LATEST MISHAPS
  03 Apr 2026  San Juan, Batangas Weather model (ERA5)         no hazards
  24 Feb 2026  TOG 8     

## How it works

1. **Place.** Each mishap's *Location* is matched to the Places list, e.g. `EAAB` → Edwin Andrews Air Base.
2. **Airfield.** The nearest airfield with a report archive is picked. If it's within the distance you set,
   its real reports are used (**observed**). If not, a weather model gives an **estimate**.
3. **Window.** With a recorded *Time*, only reports within ±2 hours count; otherwise the whole day
   (Philippine time).
4. **Hazards.** These are the same rules as the live airfield weather on the Forecast page: thunderstorms or lightning, visibility under
   about 5 km, gusts of 25 kt or more (*brief crews*); CB clouds, wind of 20 kt or more, haze, rain (*be aware*).
5. **Comparison.** For each hazard:
   *mishap days* = share of our mishap days (at airfields within range) that had it;
   *usual* = share of **all** days at those same airfields that had it.
   A binomial test says whether the gap is bigger than chance (p under 0.05). Under 10 mishap days → *too few to tell*.

**Reading it:** "same as usual" means bad weather was there about as often as on any other day, so
weather isn't a stand-out factor in our record overall. It can still matter in a single case: the
board's report decides that.


## Look closer (optional)

In [3]:
# @title Every mishap and its weather
# @markdown Run the watcher first. Shows each record with the airfield used and what was reported.
show = RECORDS[["id", "mishap_date", "mishap_time", "location", "environment", "category"]].merge(
    LINKS, left_on="id", right_on="mishap_id")
show["hazards"] = show["hazards"].map(lambda hz: ", ".join(h["text"] for h in hz))
show[["mishap_date", "mishap_time", "location", "environment", "category", "source", "station",
      "distance_km", "level", "hazards", "note"]].sort_values("mishap_date", ascending=False)


,mishap_date,mishap_time,location,environment,category,source,station,distance_km,level,hazards,note
127,2026-04-03,None,"San Juan, Batangas",ground,Vehicle / road collision,estimate,NaN,0.0,clear,,"Nearest airfield report is Manila (87 km), so ..."
126,2026-02-24,None,TOG 8,flight,Landing gear / tire / brake,estimate,NaN,0.0,aware,Rain / showers (model),Nearest airfield report is Mactan-Cebu (154 km...
125,2026-02-18,None,MDAAB,flight,Bird / wildlife strike,observed,RPLL,11.0,clear,,24 reports that day · lowest visibility 10 km ...
124,2026-02-12,None,LAB,flight,Bird / wildlife strike,estimate,NaN,0.0,aware,Rain / showers (model),"Nearest airfield report is Dumaguete (177 km),..."
123,2026-02-01,None,TOG 8,flight,Kite / wire strike,estimate,NaN,0.0,aware,Heavy rain (model),Nearest airfield report is Mactan-Cebu (154 km...
...,...,...,...,...,...,...,...,...,...,...,...
5,2016-05-04,None,Butuan,flight,Bird / wildlife strike,estimate,NaN,0.0,aware,Rain / showers (model),"Nearest airfield report is Davao (203 km), so ..."
3,2016-04-18,None,EAAB,flight,Landing gear / tire / brake,observed,RPMZ,0.0,aware,CB clouds nearby,16 reports that day · lowest visibility 10 km ...
2,2016-03-29,None,EAAB,flight,Landing gear / tire / brake,observed,RPMZ,0.0,aware,CB clouds nearby,17 reports that day · lowest visibility 10 km ...
1,2016-03-19,None,MDAAB,flight,Bird / wildlife strike,observed,RPLL,11.0,clear,,24 reports that day · lowest visibility 10 km ...


In [4]:
# @title Ordinary days at each airfield
# @markdown How often each hazard shows up on any day, the "usual" in the comparison.
pd.DataFrame({sid: {"days with data": len(d), **{label: f"{d[key].mean():.0%}" for key, label in COMPARE}}
              for sid, d in DAILY.items()}).T


,days with data,Any brief-level weather,Thunderstorms,Low visibility (under 5 km)
RPLB,3876,48%,38%,10%
RPLC,3535,33%,25%,17%
RPLL,3876,33%,26%,8%
RPMZ,3875,25%,22%,4%
RPVM,3871,42%,39%,6%
